<div class="guide-start-hint">

**Start here · Your kit and your data.** Click into the cell below and press **Shift + Enter** to open it: the kit's state, then your data: datasets, your own files and tables, and external sources. Its **Guided / Code** switch, top right, shows the code behind this page.

</div>

In [ ]:
import exakit.guide as guide

guide.start()

# The code behind this page

<div class="guide-code-start"></div>

The same steps as the cards above, as cells you can read, change and run one by one. **Code**, at the top right of a card, opens this section; **Guided** closes it again.

## Start here: your kit and your data

**Dataset:** `amazon_sales` (default), `fraud_transactions` or your own file.

Steps 0 Check the kit and 1 Access. All steps: Access, Understand, Prepare, Explore, Model, Validate, Ask with AI, Share, Hand off

At the end you have a connection and your data, from as many sources as you like. `01_explore` opens it to understand it: the column map and the semantic views. No AI model is needed. Kit commands run from these cells, so you do not need a terminal.

## Step 0 · Check the kit

Shows what is installed and running: the database, each add-on and the loaded datasets. It changes nothing.

The next cells read the rest of the kit live, change it from here (each with its log in the cell), and set up your AI model and Langfuse tracing. They are the calls the Home card's rows and buttons make.

In [ ]:
import exakit.nb as nb

kit_state = nb.kit_check()

In [ ]:
components = nb.components()
addons = nb.addons()
ai_clients = nb.ai_clients()
sample_datasets = nb.sample_datasets()
loaded_files = nb.loaded_files()
sources = nb.sources()
notebooks = nb.notebooks("data-scientist")
persona_plan = nb.persona_plan()
approved = nb.approved_datasets()
workflows = nb.list_workflows()
workflow_events = nb.workflow_log(workflows[-1]["id"]) if workflows else []
model, tracing = nb.model_settings(), nb.tracing()
[(c["id"], c["installed"], c["status"]) for c in components]

| Setting | What it does |
|---|---|
| `START_DATABASE` | start the database after a restart of this computer |
| `ADD_ON` | an add-on to install and start: one of `[a["id"] for a in addons]` |
| `LOAD`, `LOAD_SCHEMA` | a kit dataset, a file or a folder to load, and the schema it lands in |
| `SERVICE` | a service to start, then its log |
| `CHECK_AI` | check the AI clients' connection to the database |
| `UPDATE_KIT` | bring the kit and its add-ons up to date |

In [ ]:
START_DATABASE = False
ADD_ON = ""
LOAD = ""
LOAD_SCHEMA = ""
SERVICE = ""
CHECK_AI = False
UPDATE_KIT = False
if START_DATABASE:
    nb.exakit("start")
if ADD_ON:
    nb.addon_ready(ADD_ON)
if LOAD:
    nb.exakit("data-load", LOAD, env={"EXAKIT_SCHEMA": LOAD_SCHEMA.upper()} if LOAD_SCHEMA else None)
if SERVICE:
    nb.exakit("start", SERVICE)
    nb.exakit("logs", SERVICE)
if CHECK_AI:
    nb.exakit("mcp-doctor")
if UPDATE_KIT:
    nb.kit_update()

| Setting | What it does |
|---|---|
| `PROVIDER` | one of `providers`: `"anthropic"`, `"openai"`, `"gemini"`, `"ollama"` |
| `SHOW_MODELS` | list the models your key may use; the key is asked in a hidden box |
| `MODEL_ID`, `EFFORT` | the model, and its effort where it takes one |
| `PROVIDER_BASE_URL` | empty: the provider's own address |
| `LANGFUSE_BASE_URL` | where model calls are traced, such as `https://cloud.langfuse.com` |
| `FORGET` | delete what was remembered |

In [ ]:
providers = {p["id"]: p for p in nb.model_providers()}
PROVIDER = ""
SHOW_MODELS = False
MODEL_ID = ""
EFFORT = ""
PROVIDER_BASE_URL = ""
LANGFUSE_BASE_URL = ""
FORGET = False
if PROVIDER and not PROVIDER_BASE_URL:
    PROVIDER_BASE_URL = providers[PROVIDER]["base_url"]
if PROVIDER and SHOW_MODELS:
    print(nb.provider_models(PROVIDER, base_url=PROVIDER_BASE_URL))
if MODEL_ID:
    nb.use_model(MODEL_ID, PROVIDER_BASE_URL, effort=EFFORT, remember=True)
if LANGFUSE_BASE_URL:
    nb.use_model(langfuse_base_url=LANGFUSE_BASE_URL, remember=True)
if FORGET:
    nb.forget_model(model=True, langfuse=True)

## Step 1 · Access the data

Starts a workflow, connects to the database and loads a dataset: a practice dataset, your own file or a connected source.

### Start a workflow

The workflow gives one ID to every query and decision below. The next notebooks continue it. You do not copy anything.

In [ ]:
import os
import exakit.nb as nb

workflow_id = nb.workflow("data-scientist-quickstart", resume=os.environ.get("EXAKIT_WORKFLOW_ID"))
print(f"Workflow {workflow_id}: every step below is recorded under it.")

### Connect

Connects with the kit's saved connection and lists the schemas. The notebook holds no password.

In [ ]:
import exakit.nb as nb

connection = nb.connect()
nb.check_connection(connection)

In [ ]:
nb.list_schemas(connection)

### Choose a dataset

These notebooks use two practice datasets:

- `amazon_sales`: online fashion orders, one row per order line.
- `fraud_transactions`: card and bank payments with a fraud label. It has repeats, gaps and placeholders on purpose.

Set `SAMPLE` to one of them. The first run downloads it (internet needed). Later runs reuse it. Keep `SOURCE = None` for this walkthrough.

In [ ]:
SAMPLE = "amazon_sales"
sample_data = nb.load_sample(connection, SAMPLE)

**Load your own data (optional):** set `MY_DATA` to a CSV, Parquet or JSON file, or to a folder of them. Empty skips it.

Each file becomes one table named after the file, in a schema named after its folder (`MY_SCHEMA` names another). A folder loads like `exakit data-load`. Running the cell again skips what is already loaded. `MY_TABLE` uses a table already in the database instead. `LOAD_RULES` changes how a file is loaded, such as `{"REPLACE": True, "DATE_SHARE": 0.8}`; `nb.load_rules()` lists every rule.

In [ ]:
MY_DATA = ""
MY_SCHEMA = ""
MY_TABLE = ""
LOAD_RULES = {}
own_data = (nb.use_table(connection, MY_TABLE) if MY_TABLE
            else nb.load_own_file(MY_DATA, MY_SCHEMA, conn=connection, missing_ok=True, rules=LOAD_RULES))

**An external source (optional):** a table in Postgres, MySQL, Snowflake and others, read live through a virtual schema. Nothing is copied.

1. Not connected yet? Pick `CONNECTOR` and paste its connection string into `SOURCE_URL`: the kit reads it as `exakit connect --url` does, and `source_reading["missing"]` says what it did not give. Put those (and any other input you want to change, by the names in `connector_forms[CONNECTOR]["inputs"]`) in `SOURCE_INPUTS`. No connection string? Leave it empty and give every input in `SOURCE_INPUTS`. On a Mac, use `192.168.64.1` for a database on this computer.
2. Test it: the kit checks the inputs and this computer tries the source's host and port. Nothing is changed and no password is needed.
3. Connect it: the password is asked in a hidden box and goes to the kit only.
4. Set `SOURCE` to its table, `"VS_<NAME>.<TABLE>"`. `"VS_<NAME>"` alone lists the tables. `None` skips it.

| Setting | What it does |
|---|---|
| `CONNECTOR` | the kind of source: `[c["id"] for c in connectors]`, such as `"postgresql"` or `"snowflake"` |
| `SOURCE_NAME` | its name: the tables appear as `VS_<NAME>` |
| `SOURCE_SETTINGS` | for a source with several kinds in one, such as `{"catalog": "unity", "files": "s3"}` |
| `SOURCE_URL` | its connection string; a password in it is used, never shown |
| `SOURCE_INPUTS` | what the string did not give, such as `{"user": "reader"}` |

In [ ]:
connectors = nb.connectors()
connector_forms = {c["id"]: nb.connector_inputs(c["id"]) for c in connectors}
source_tips = nb.source_engine("catalog", "--public-ip")["events"]
CONNECTOR = ""
SOURCE_NAME = ""
SOURCE_SETTINGS = {}
if CONNECTOR and SOURCE_SETTINGS:
    connector_forms[CONNECTOR] = nb.connector_inputs(CONNECTOR, **SOURCE_SETTINGS)
SOURCE_URL = ""
source_reading = nb.read_connection_string(CONNECTOR, SOURCE_URL, **SOURCE_SETTINGS) if CONNECTOR and SOURCE_URL else None
SOURCE_INPUTS = {}
SOURCE_INPUTS = {**SOURCE_SETTINGS, **({"url": SOURCE_URL} if SOURCE_URL else {}), **SOURCE_INPUTS}

In [ ]:
source_test = nb.test_source(CONNECTOR, SOURCE_NAME, **SOURCE_INPUTS) if CONNECTOR else None

In [ ]:
connected = nb.connect_source(CONNECTOR, SOURCE_NAME, **SOURCE_INPUTS) if CONNECTOR else None

In [ ]:
SOURCE = None
source_data = nb.load_source(connection, SOURCE or connected)

The notebook uses the source first, then your file, then the sample.

In [ ]:
chosen_data = nb.first(source_data, own_data, sample_data)

### All your data in one map

Every table by origin: external source, your data, sample data. Each shows its rows, its column map and the roles in it. *Links* lists columns that join tables, across origins too. A `possible` link only shares a name: check it before a join.

In [ ]:
nb.data_map(connection)

### Carry the choice to the next notebook

`01_explore` opens the tables chosen here: their column maps and semantic views are its first step. Several tables? Put them all in: `nb.remember_chosen(chosen_tables)`.

`OPEN_APPROVED`: a dataset approved before, by its id (`nb.approved_datasets()` lists them).

In [ ]:
OPEN_APPROVED = ""
if OPEN_APPROVED:
    chosen_data = nb.dataset(connection, OPEN_APPROVED)
remembered = nb.remember_chosen(chosen_data)
print("01_explore opens: " + ", ".join(remembered))

## Next

Open `01_explore` to understand your data - the column map and the semantic views - then prepare, explore, model and validate (steps 2 to 6). Then open `02_ask_your_data` (step 7) and `03_share` (steps 8 and 9). Each one continues this workflow.